# Predicting a synthetic stock from news headlines

Making money in trading is about being fast and smart. Excellent coding covers the "fast" part, but how can we be "smart"?

The goal of Algo is to make **hypotheses** about the world and to be able to **evaluate** them. Whenever we have a hypothesis, the first step is almost always going to be to look at the data.

In this challenge, the market is simulated. Each **session** is one synthetic stock trading for 100 time **bars**. Along the way, news headlines come out. The headlines are fictional and so are the companies they mention.

You see the first half of each session: bars 0 to 49 and the headlines published during them. At the close of bar 49 you take a position, and you hold it for 10 bars, until the close of bar 59. Your job is to predict the stock's return over those 10 bars.

The returns are **market-adjusted**: we subtract the average return across all sessions, so a session's return is positive if it did better than a typical session. Betting that every stock goes up earns nothing here.

## Setup
You should make a copy of this notebook. The first cell will download the data and source needed.

In [2]:
!pip -q install gdown

FOLDER_URL = "https://drive.google.com/drive/folders/1mYOtE9R-XtdcJ4cPVpQYAxiPLD1Dm4vX?usp=sharing"
!gdown --folder "$FOLDER_URL" --output /content

Retrieving folder contents
Processing file 1UP6Yr4XYHl9iaqX9BlbuGaVpcp1Os2xo ins_bars.parquet
Processing file 1E_1TrSYdXOEiNxjTRxCOT0XQX65gxmn7 ins_headlines.parquet
Processing file 1yep_-avWzvVxCRIo4CNkjiZyCv7i2p09 ins_returns.parquet
Processing file 1MPaC1U8V2GUZrpAAwB2RpuLdRu2uVcBi oos_bars.parquet
Processing file 1cmNZbk5ILzz_grgFYcxEpj1HToaoIQ2M oos_headlines.parquet
Processing file 1JEVRT8U7eD1XKSzs0PcI1aP3QI9sMJKz sessions.py
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1UP6Yr4XYHl9iaqX9BlbuGaVpcp1Os2xo
To: /content/ins_bars.parquet
100% 5.84M/5.84M [00:00<00:00, 28.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1E_1TrSYdXOEiNxjTRxCOT0XQX65gxmn7
To: /content/ins_headlines.parquet
100% 2.96M/2.96M [00:00<00:00, 174MB/s]
Downloading...
From: https://drive.google.com/uc?id=1yep_-avWzvVxCRIo4CNkjiZyCv7i2p09
To: /content/ins_returns.parquet
100% 155k/155k [00:00

In [5]:
import sessions # pyright: ignore[reportMissingImports]

ModuleNotFoundError: No module named 'rich'

# Part 1. Data exploration

The first section of this notebook loads the data and does some basic exploration. You should run each cell. Feel free to add and run additional cells to explore the data further.

## Get the data

You have 10,000 in-sample sessions for training and 10,000 out-of-sample sessions to predict.

Make sure the `ins_*.parquet` and `oos_*.parquet` files are in the same directory as this notebook and `sessions.py`.

In [ ]:
df = sessions.get_ins_data()

NameError: name 'sessions' is not defined

In [4]:
df.head()

NameError: name 'df' is not defined

Each row is one in-sample session:

- `open`, `high`, `low`, `close`: arrays of 50 prices, one per bar from 0 to 49. Prices start at 1.
- `headlines`: the headlines published during bars 0 to 49, in order.
- `headline_bars`: the bar each of those headlines was published in.
- `returns`: the market-adjusted return from the close of bar 49 to the close of bar 59. This is what you predict.

## What does a session look like?

In [1]:
session = df.iloc[0]
print(session)
for bar, headline in zip(session["headline_bars"], session["headlines"], strict=True):
    print(bar, headline)

NameError: name 'df' is not defined

In [ ]:
import matplotlib.pyplot as plt

plt.plot(session["close"])
plt.xlabel("Bar")
plt.ylabel("Close price")

## Visualize the distribution of returns

In [ ]:
df["returns"].hist(bins=50)

## The whole session, for research

In-sample, you can also see the second half of every session: bars 50 to 99 and their headlines. Out-of-sample you will only have the first half, so your prediction can't use these. They are still useful for research, for example to look at what prices do after a headline.

In [ ]:
full_bars = sessions.get_ins_full_bars()
full_headlines = sessions.get_ins_full_headlines()
full_headlines.head()

# Part 2. Predicting returns

The rest of this notebook is dedicated to the actual task of predicting returns. It explains the task and evaluation, and also gives some basic ideas and useful utility functions.

## Your task

Your goal is to write a function that takes a row of `df` (minus the `returns` column) and returns a prediction of the return from the close of bar 49 to the close of bar 59.
Here is an example of a valid submission, based on the naive idea that "stocks go up".

```python
def my_prediction(row: pd.Series) -> float:
    return 1.0
```

You can call `sessions.get_ins_predictions` to see your predictions on the in-sample dataset: this will be provided as the new column `"pred"` in your dataframe.

In [ ]:
def my_prediction(row):
    return 1.0  # stocks go up!

In [ ]:
df_predictions = sessions.get_ins_predictions(my_prediction)
df_predictions[["headlines", "returns", "pred"]].head()

## How are you evaluated?

Your goal is to produce a prediction that makes money AS CONSISTENTLY AS POSSIBLE.
We have implemented a very simple trader that will use your prediction to trade. Your score is the *Sharpe ratio* of this trader's performance.

Here is how the trader behaves:
* At the close of bar 49, it takes a position equal to your prediction: it buys if your prediction is positive and sells if it is negative, more for a larger prediction.
* It holds the position until the close of bar 59 and receives `prediction * return`.
* Each session counts as one trading day, so the Sharpe ratio is `sqrt(252) * mean(session_profit) / stddev(session_profit)`. This is a common metric we use in Algo to evaluate our strategies.

## Running an in-sample backtest

We can use your model's in-sample performance to *estimate* how well it will perform out-of-sample. We use these in-sample *backtests* to evaluate our trading hypotheses. The function `sessions.run_ins_backtest` calls your prediction model in-sample and computes the Sharpe ratio (as well as several other metrics).

In [ ]:
sessions.run_ins_backtest(my_prediction)

Because returns are market-adjusted, "stocks go up" has a Sharpe ratio of about zero. You'll have to do better!

Here's how to read these stats:

- **Sharpe** is the in-sample Sharpe ratio of your prediction model.
- **Total PnL** is the amount of money that your strategy made.
- **Frac nonzero prediction** is the fraction of sessions where your model issued a nonzero prediction.
- **Correlation** is the correlation between your prediction and the returns.
- **Mean return positive** is the average return among sessions for which you issued a positive forecast. Ideally, this number would be positive.
- **Mean return negative** is the average return among sessions for which you issued a negative forecast. Ideally, this number would be negative.
- **Mean daily portfolio size** is the average size of your position. Roughly, it measures the amount of capital you need to actually implement your strategy.
- **Annual return** is the ratio of your total PnL to the total size of your positions. In practice, a strategy that makes a lot of money (total PnL) but has a low annual return is hard to implement due to the large capital requirements.

## Getting ready to go out-of-sample

Once you think you have a good submission, you can run it on the out-of-sample sessions. These are the predictions you will submit!

In [ ]:
df_oos_predictions = sessions.get_oos_predictions(my_prediction)
df_oos_predictions.head()

### Scoring

How good is your model? Well, you can't run a backtest, because the out-of-sample returns are secret!

Instead, `sessions.write_submission(...)` writes your out-of-sample predictions to `predictions.csv`. Arcade calls its columns `headline_id,pred`; `headline_id` holds the session number. **Upload that file on the challenge page in arcade**, which scores it against the hidden out-of-sample returns. Your out-of-sample Sharpe appears on the live leaderboard.

Be aware that you only get **10 uploads**, so don't waste them! Use in-sample backtests to decide what is worth submitting.

In [ ]:
# This will not work!
# sessions.run_oos_backtest(my_prediction)

## Ideas to get you started

Here are some ideas you may wish to explore in your submission. Of course, you are not required to pursue these ideas.

**Looking at keywords**. Some headlines sound like good news and some sound like bad news. Maybe a session with more good news does better?

In [ ]:
def keyword_predictor(row):
    score = 0.0
    for headline in row["headlines"]:
        if "secures" in headline:
            score += 1.0
        if "decline" in headline:
            score -= 1.0
    return score

In [ ]:
sessions.run_ins_backtest(keyword_predictor)

In [ ]:
df_oos_predictions = sessions.get_oos_predictions(keyword_predictor)

In [ ]:
sessions.write_submission(
    prediction_df=df_oos_predictions,
    path="predictions.csv",
)
# Now upload predictions.csv on the challenge page in arcade.